# Week 4 · Class 2 — Gradio Capstone UI

Take the Class 1 Constitution RAG pipeline and wrap it in a **Gradio** interface:

- Chat answers grounded in retrieved passages
- Source panel with scores / pages
- Sliders for **top-k** and **score threshold**

## Learning goals

1. Rebuild the RAG core as reusable functions (self-contained Colab)
2. Wire retrieve + answer into Gradio Blocks
3. Demo a shareable Capstone UI

## Before you start

Same Groq key + `nepal-constitution.pdf` as Class 1.

## Section 1 — Install packages

In [ ]:
!pip install -q langchain-core langchain-community langchain-text-splitters langchain-groq langchain-huggingface qdrant-client pypdf sentence-transformers gradio

## Section 2 — Groq API key

In [ ]:
import os
import getpass

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

print("Key loaded:", bool(os.environ.get("GROQ_API_KEY")))

## Section 3 — Rebuild the RAG pipeline

Class 2 is **self-contained** — we rebuild the in-memory index here (Colab runtimes do not keep Class 1 state).

In [ ]:
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)

CANDIDATES = [
    Path("nepal-constitution.pdf"),
    Path("data/nepal-constitution.pdf"),
    Path("../data/nepal-constitution.pdf"),
    Path("/content/nepal-constitution.pdf"),
    Path("/content/data/nepal-constitution.pdf"),
]
PDF_PATH = next((p for p in CANDIDATES if p.exists()), None)
if PDF_PATH is None:
    raise FileNotFoundError("Upload nepal-constitution.pdf (see week-4/data/README.md)")

pages = PyPDFLoader(str(PDF_PATH)).load()
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(pages)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

COLLECTION = "nepal_constitution"
client = QdrantClient(":memory:")
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)
texts = [c.page_content for c in chunks]
vectors = embeddings.embed_documents(texts)
client.upsert(
    collection_name=COLLECTION,
    points=[
        PointStruct(
            id=i,
            vector=vectors[i],
            payload={
                "text": chunks[i].page_content,
                "source": str(chunks[i].metadata.get("source", PDF_PATH.name)),
                "page": chunks[i].metadata.get("page"),
            },
        )
        for i in range(len(chunks))
    ],
)
print(f"Ready: {len(chunks)} chunks from {PDF_PATH.name}")

## Section 4 — `retrieve` + `ask` with tunable dials

In [ ]:
docs_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions about the Constitution of Nepal. "
     "Use ONLY the retrieved context below. Cite [page N] when available. "
     "If context is insufficient, say you do not find that in the constitution."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

def retrieve(question: str, k: int = 4, score_threshold: float = 0.25):
    hits = client.query_points(
        collection_name=COLLECTION,
        query=embeddings.embed_query(question),
        limit=int(k),
    ).points
    return [
        {
            "text": h.payload["text"],
            "source": h.payload.get("source"),
            "page": h.payload.get("page"),
            "score": float(h.score),
        }
        for h in hits
        if h.score >= float(score_threshold)
    ]

def format_hits(hits):
    blocks = []
    for h in hits:
        tag = f"page {h['page']}" if h.get("page") is not None else h.get("source", "doc")
        blocks.append(f"**[{tag} | score={h['score']:.3f}]**\n{h['text']}")
    return "\n\n".join(blocks) if blocks else "_No chunks above the score threshold._"

def ask(question: str, k: int = 4, score_threshold: float = 0.25):
    hits = retrieve(question, k=k, score_threshold=score_threshold)
    if not hits:
        answer = "I could not find that in the constitution (no chunk passed the score threshold)."
    else:
        answer = (docs_prompt | llm).invoke({
            "question": question,
            "context": format_hits(hits),
        }).content
    return answer, hits

answer, hits = ask("What rights relate to equality?")
print(answer)
print("hits:", len(hits))

## Section 5 — Gradio Blocks layout

Layout goals:

1. **Chat** — user questions + grounded answers
2. **Sources** — markdown snippets with page + score
3. **Controls** — sliders for `k` and score threshold

In [ ]:
import gradio as gr

def respond(message, history, k, score_threshold):
    answer, hits = ask(message, k=k, score_threshold=score_threshold)
    sources_md = format_hits(hits)
    # Gradio Chatbot messages format (role/content dicts)
    history = history or []
    history = list(history) + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer},
    ]
    return history, sources_md

with gr.Blocks(title="Nepal Constitution RAG") as demo:
    gr.Markdown("# Nepal Constitution RAG\nGrounded answers from an in-memory Qdrant index.")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=420, label="Chat")
            msg = gr.Textbox(label="Ask the constitution", placeholder="e.g. What does it say about equality?")
            with gr.Row():
                k = gr.Slider(1, 8, value=4, step=1, label="top-k")
                threshold = gr.Slider(0.0, 0.8, value=0.25, step=0.05, label="score threshold")
            clear = gr.Button("Clear")
        with gr.Column(scale=2):
            sources = gr.Markdown("_Retrieved sources will appear here._", label="Sources")

    msg.submit(respond, [msg, chatbot, k, threshold], [chatbot, sources]).then(
        lambda: "", None, msg
    )
    clear.click(lambda: ([], "_Retrieved sources will appear here._"), None, [chatbot, sources])

demo.launch(share=False)

### Tips

- In Colab, `demo.launch()` prints a local URL; set `share=True` for a temporary public link
- If answers feel vague, raise `k` or lower the threshold slightly — then check the Sources panel
- If off-topic questions slip through, raise the threshold

## Section 6 — Capstone demo script (2–3 min)

1. Ask an on-constitution question → show answer + Sources
2. Move the threshold slider → show how retrieval changes
3. Ask something off-constitution → honest miss

## Deliverable checklist

- [ ] Gradio chat answers from Qdrant-retrieved chunks only
- [ ] Sources panel shows text + page/score
- [ ] `k` and score-threshold sliders work live
- [ ] Demo ready for class